# Remote CNOT on a line: compare swaps with measured ancillas

Implement a CNOT between the ends of a four-qubit line with two different resources: reversible swaps, or an entangled ancilla pair plus measurement and feedforward. Verify the complete two-qubit channel using its Choi state, rather than checking just one Bell-state input. You need density matrices, partial trace, and classical conditional operations.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Separate data qubits from transport ancillas

Data live on $q_0$ (control) and $q_3$ (target); ancillas $q_1,q_2$ start at zero. Prepare a Bell pair on the ancillas, couple $q_0\to q_1$, measure $q_1$ as $m_1$, and apply $X^{m_1}$ on $q_2$. Then couple $q_2\to q_3$, measure $q_2$ in the X basis as $m_2$, and apply $Z^{m_2}$ on $q_0$. All two-qubit gates use adjacent qubits. The ancillas are consumed; unlike the swap baseline, the protocol does not preserve arbitrary initial ancilla states.


In [ ]:
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import DensityMatrix, Operator, Statevector, partial_trace, state_fidelity
from qiskit_aer import AerSimulator


def swap_remote_cx(qc):
    qc.swap(0, 1)
    qc.swap(1, 2)
    qc.cx(2, 3)
    qc.swap(1, 2)
    qc.swap(0, 1)
    return qc


def measured_remote_cx(qc, correct=True):
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.measure(1, 0)
    if correct:
        with qc.if_test((qc.clbits[0], 1)):
            qc.x(2)
    qc.cx(2, 3)
    qc.h(2)
    qc.measure(2, 1)
    if correct:
        with qc.if_test((qc.clbits[1], 1)):
            qc.z(0)
    return qc


unitary = swap_remote_cx(QuantumCircuit(4))
reference = QuantumCircuit(4)
reference.cx(0, 3)
assert np.allclose(Operator(unitary).data, Operator(reference).data, atol=1e-12)
print(measured_remote_cx(QuantumCircuit(4, 2)).draw(output="text"))

## 2. Use a Choi-state test to cover all inputs

Checking $|+0\rangle$ alone could miss an incorrect phase correction. Instead, maximally entangle each data qubit with a reference: $(q_0,q_4)$ and $(q_3,q_5)$. Apply the channel only to the data and compare the reduced data-plus-reference density matrix with an ideal CX. Equality of these normalized Choi states certifies equality of the channels in ideal simulation. Reference-state preparation is a test fixture, outside the line-protocol resource count.


In [ ]:
def choi_input(classical_bits=0):
    qc = QuantumCircuit(6, classical_bits)
    qc.h(0)
    qc.cx(0, 4)
    qc.h(3)
    qc.cx(3, 5)
    return qc


ideal = choi_input()
ideal.cx(0, 3)
expected = partial_trace(Statevector.from_instruction(ideal), [1, 2])
sim = AerSimulator(method="density_matrix")


def run_channel(correct):
    qc = measured_remote_cx(choi_input(2), correct=correct)
    qc.save_density_matrix()
    compiled = transpile(qc, sim, optimization_level=0, seed_transpiler=31)
    result = sim.run(compiled, shots=4096, seed_simulator=31).result()
    return partial_trace(DensityMatrix(result.data(0)["density_matrix"]), [1, 2])


actual = run_channel(True)
without_feedforward = run_channel(False)
fidelity = state_fidelity(actual, expected)
assert fidelity > 1 - 1e-10
assert np.max(np.abs(actual.data - expected.data)) < 1e-10
assert state_fidelity(without_feedforward, expected) < 0.4
print(
    {
        "corrected_Choi_fidelity": fidelity,
        "uncorrected_Choi_fidelity": state_fidelity(without_feedforward, expected),
    }
)

## 3. Count comparable resources

Compile only the protocol bodies into a CX plus single-qubit basis. The swap baseline has 13 CX gates: four swaps at three CX each, plus the logical CX. The measured protocol has three CX gates and two measurements, with conditional corrections. These counts do not establish a timing advantage: mid-circuit measurement, classical communication, reset, and calibration all affect hardware latency and fidelity.


In [ ]:
baseline = transpile(
    swap_remote_cx(QuantumCircuit(4)), basis_gates=["cx", "u"], optimization_level=0
)
dynamic = measured_remote_cx(QuantumCircuit(4, 2))
assert baseline.count_ops().get("cx", 0) == 13
assert dynamic.count_ops().get("cx", 0) == 3
assert dynamic.count_ops().get("measure", 0) == 2
for instruction in dynamic.data:
    if instruction.operation.name == "cx":
        a, b = [dynamic.find_bit(q).index for q in instruction.qubits]
        assert abs(a - b) == 1
print(
    pd.DataFrame(
        [
            {
                "protocol": "swap",
                "CX": 13,
                "mid-circuit measurements": 0,
                "classical conditionals": 0,
            },
            {
                "protocol": "measured ancillas",
                "CX": 3,
                "mid-circuit measurements": 2,
                "classical conditionals": 2,
            },
        ]
    )
)

## Limits, cleanup, and next steps

The density-matrix simulator has no measurement latency or decoherence. This four-qubit protocol is a concrete building block, not a demonstration of constant-depth performance at arbitrary distance. Hardware execution requires an explicitly chosen backend supporting dynamic circuits, a validated layout, current calibration data, an account with adequate quota, and a separately reviewed job submission. No cloud SDK or hardware call is hidden here.

No artifacts or jobs were created. Release simulation memory by restarting the kernel. Extend the channel test to a noisy model and compare fidelity at matched wall-clock assumptions.

Primary references: [IBM long-range entanglement tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/long-range-entanglement), [Bäumer et al., Efficient Long-Range Entanglement Using Dynamic Circuits](https://arxiv.org/abs/2308.13065), [Qiskit dynamic circuits guide](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow).

**Attribution:** This independent rewrite develops topics from Qiskit Global Summer School 2025. See [course attribution and Apache 2.0 notice](../ATTRIBUTION.md). No course artwork, grading service, saved account, or hardware results are included. This is not an official IBM publication and does not imply IBM endorsement.
